# Simple Test Fixture Generation (v2)

This notebook creates test fixtures using the utilities.py module with python-ffmpeg wrapper.

**Key Improvements:**
- Uses python-ffmpeg wrapper (same as working minutia code)
- Properly extracts all audio tracks from MKV files
- Modular utilities for reuse
- Creates small, fast test clips (5-15 seconds)

**Generated Fixtures:**
- `basic_5s.mkv` - Clear speech, good audio quality
- `noisy_10s.mkv` - Background noise, harder transcription
- `complex_8s.mkv` - Mixed audio content

**Output Directory:** `tests/fixtures/`

## Setup

In [ ]:
from pathlib import Path
from utilities import (
    list_mkv_recordings,
    create_test_fixture,
    save_fixture_metadata
)

# Paths
recordings_dir = Path('../../+recordings')
fixtures_dir = Path('../fixtures')

# Create base directory
fixtures_dir.mkdir(parents=True, exist_ok=True)

print(f"Recordings source: {recordings_dir}")
print(f"Fixtures output: {fixtures_dir}")

## Available Source Recordings

In [ ]:
# List available recordings with metadata
mkv_files = list_mkv_recordings(recordings_dir)

if not mkv_files:
    print("⚠ No MKV files found! Make sure recordings are in +recordings/")

## Create Test Fixtures

Generate small test clips from available recordings

In [ ]:
# Track all created fixtures
fixtures_metadata = []

# Fixture 1: Basic clear speech (5 seconds)
if recordings_dir / 'Example-Microphone.mkv' in mkv_files:
    metadata = create_test_fixture(
        source_mkv=recordings_dir / 'Example-Microphone.mkv',
        output_name='basic_5s',
        fixtures_dir=fixtures_dir,
        start_seconds=1.0,
        duration_seconds=5.0,
        description='Clear microphone audio, 5 seconds for fast testing'
    )
    fixtures_metadata.append(metadata)
else:
    print("⚠ Example-Microphone.mkv not found, skipping basic fixture")

In [ ]:
# Fixture 2: Noisy audio (10 seconds)
if recordings_dir / 'Example-Noisy.mkv' in mkv_files:
    metadata = create_test_fixture(
        source_mkv=recordings_dir / 'Example-Noisy.mkv',
        output_name='noisy_10s',
        fixtures_dir=fixtures_dir,
        start_seconds=2.0,
        duration_seconds=10.0,
        description='Background noise and audio artifacts, harder transcription'
    )
    fixtures_metadata.append(metadata)
else:
    print("⚠ Example-Noisy.mkv not found, skipping noisy fixture")

In [ ]:
# Fixture 3: Complex audio (8 seconds)
if recordings_dir / 'Example-Song+MLKSpeech.mkv' in mkv_files:
    metadata = create_test_fixture(
        source_mkv=recordings_dir / 'Example-Song+MLKSpeech.mkv',
        output_name='complex_8s',
        fixtures_dir=fixtures_dir,
        start_seconds=3.0,
        duration_seconds=8.0,
        description='Music and speech overlapping, complex audio scenario'
    )
    fixtures_metadata.append(metadata)
else:
    print("⚠ Example-Song+MLKSpeech.mkv not found, skipping complex fixture")

In [ ]:
# Fixture 4: Podcast-style (12 seconds) 
if recordings_dir / 'Example-Podcast.mkv' in mkv_files:
    metadata = create_test_fixture(
        source_mkv=recordings_dir / 'Example-Podcast.mkv',
        output_name='podcast_12s',
        fixtures_dir=fixtures_dir,
        start_seconds=5.0,
        duration_seconds=12.0,
        description='Clear speech, podcast-style conversation'
    )
    fixtures_metadata.append(metadata)
else:
    print("⚠ Example-Podcast.mkv not found, skipping podcast fixture")

## Summary and Metadata

In [ ]:
print(f"\n{'='*60}")
print(f"FIXTURE GENERATION SUMMARY")
print(f"{'='*60}")

if fixtures_metadata:
    print(f"✅ Created {len(fixtures_metadata)} test fixtures:")
    
    for fixture in fixtures_metadata:
        if 'error' not in fixture:
            name = fixture['name']
            desc = fixture['description']
            duration = fixture['validation'].get('duration', 'unknown')
            tracks = fixture['validation'].get('audio_tracks', 'unknown')
            print(f"  - {name}: {desc}")
            print(f"    Duration: {duration:.1f}s, Audio tracks: {tracks}")
        else:
            print(f"  ✗ {fixture.get('name', 'unknown')}: {fixture['error']}")
    
    # Save master metadata
    master_metadata = {
        "generated_by": "tests/scripts/generate_test_fixtures_simple_v2.ipynb",
        "description": "Simple test fixtures for fast unit/integration testing",
        "total_fixtures": len(fixtures_metadata),
        "fixtures": fixtures_metadata,
        "notes": [
            "Created with python-ffmpeg wrapper for reliability",
            "Small clips (5-12 seconds) for fast test execution",
            "Multiple audio tracks properly extracted"
        ]
    }
    
    save_fixture_metadata(fixtures_dir, master_metadata)
    
else:
    print("⚠ No fixtures were created")

print(f"\n📁 Fixtures location: {fixtures_dir}")

## Verify Created Files

In [ ]:
# List all created files
video_dir = fixtures_dir / 'video'
audio_dir = fixtures_dir / 'audio'

print("\n📹 Video fixtures:")
if video_dir.exists():
    video_files = sorted(video_dir.glob('*.mkv'))
    for vf in video_files:
        size_mb = vf.stat().st_size / (1024 * 1024)
        print(f"  {vf.name} ({size_mb:.1f} MB)")
else:
    print("  (none)")

print("\n🎵 Audio fixtures:")
if audio_dir.exists():
    for fixture_dir in sorted(audio_dir.iterdir()):
        if fixture_dir.is_dir():
            audio_files = list(fixture_dir.glob('*.wav'))
            print(f"  {fixture_dir.name}/:")
            for af in sorted(audio_files):
                size_mb = af.stat().st_size / (1024 * 1024)
                print(f"    {af.name} ({size_mb:.1f} MB)")
else:
    print("  (none)")

# Calculate total size
total_size = 0
for file in fixtures_dir.rglob('*'):
    if file.is_file() and file.suffix in ['.mkv', '.wav']:
        total_size += file.stat().st_size

print(f"\n📊 Total fixture size: {total_size / (1024 * 1024):.1f} MB")

## Next Steps

**Using these fixtures:**
1. **Unit tests** - Use individual audio files for testing transcription components
2. **Integration tests** - Use video clips for end-to-end pipeline testing
3. **Performance tests** - Small files ensure fast test execution

**Test patterns:**
```python
# Example unit test
def test_audio_transcription():
    audio_file = Path('tests/fixtures/audio/basic_5s/basic_5s_track_3.wav')
    result = transcribe_audio(audio_file)
    assert len(result.segments) > 0

# Example integration test
def test_video_extraction():
    video_file = Path('tests/fixtures/video/basic_5s.mkv')
    audio_files = extract_all_audio_tracks(video_file, temp_dir)
    assert len(audio_files) >= 2  # Should have mic + desktop
```

**For complete golden samples:**
- Run `create_complete_fixtures.ipynb` to process full recordings
- Use for regression testing against known-good outputs